# 02b · Silver → De-Identified Silver

> **SYNTHETIC DATA ONLY.** Reference pattern — not a certified de-identification service.

The **single privileged crossing point.** This notebook reads the conformed `silver_*`
tables (which still carry raw PHI: MRN, names, DOB, ZIP) and writes de-identified
`silver_deid_*` tables driven entirely by [`config/deid_rules.yaml`](../config/deid_rules.yaml).

**HARD RULE for this notebook:** never `display()`, `.show()`, `print()`, `.collect()`,
or `.toPandas()` any *raw* (pre-de-id) column. The only outputs are row counts and the
de-identified result. The tokenization **pepper** is fetched from Azure Key Vault and is
never printed, logged, or written to a table.

Run order: `01` → `02` → **`02b`** → `03b` → `NB_scorecard`.

In [ ]:
# --- make the accelerator's package importable inside Fabric (DRIVER + EXECUTORS) ---
# The package may land under Files/accelerator/ OR Files/accelerator/src/ depending on how
# it was uploaded. Auto-detect the PARENT folder that actually contains fabric_phi_deid/ so
# a mismatched upload layout can't silently break sys.path OR the executor zip below.
import os
import sys

_FILES_ROOT = "/lakehouse/default/Files/accelerator"
_SRC_CANDIDATES = [_FILES_ROOT, f"{_FILES_ROOT}/src"]
SRC_PATH = next(
    (p for p in _SRC_CANDIDATES if os.path.isdir(os.path.join(p, "fabric_phi_deid"))),
    None,
)
if SRC_PATH is None:
    raise RuntimeError(
        "fabric_phi_deid package not found. Upload src/fabric_phi_deid/ so it sits under "
        f"one of: {_SRC_CANDIDATES}"
    )
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

# Rulebook can sit next to Files/accelerator/config/ or Files/accelerator/src/config/.
CONFIG_PATH = next(
    (p for p in (f"{_FILES_ROOT}/config/deid_rules.yaml",
                 f"{SRC_PATH}/config/deid_rules.yaml") if os.path.isfile(p)),
    f"{_FILES_ROOT}/config/deid_rules.yaml",   # fall back to canonical path for a clear error
)

# --- ship the package to the EXECUTORS too (critical for the Spark UDFs) ---
# sys.path.append above only affects the DRIVER. deidentify_table() runs as UDFs on the
# executor Python workers, which do NOT inherit the driver's sys.path -> without this they
# fail with ModuleNotFoundError: No module named 'fabric_phi_deid' (surfaces as
# TASK_WRITE_FAILED at saveAsTable). Zipping the package from the DETECTED root and calling
# addPyFile distributes it to every current + future executor.
import shutil
_zip_path = shutil.make_archive("/tmp/fabric_phi_deid", "zip",
                                root_dir=SRC_PATH, base_dir="fabric_phi_deid")
spark.sparkContext.addPyFile(_zip_path)
print("Package root:", SRC_PATH, "| shipped to executors:", _zip_path)

from fabric_phi_deid import __version__ as ENGINE_VERSION             # noqa: E402
from fabric_phi_deid.tokenization import get_pepper                   # noqa: E402
from fabric_phi_deid.deid_engine import load_rules, deidentify_table  # noqa: E402
from fabric_phi_deid.config import audit_coverage                     # noqa: E402
from fabric_phi_deid.audit import (                                   # noqa: E402
    build_run_manifest, write_manifest, config_fingerprint, get_audit_logger,
)
from fabric_phi_deid.validation import scan_spark_dataframe           # noqa: E402

cfg     = load_rules(CONFIG_PATH)   # fails fast (ConfigValidationError) on a bad rulebook
PROFILE = cfg.get("active_profile", "safe_harbor")
print(f"fabric_phi_deid v{ENGINE_VERSION}  |  active profile: {PROFILE}")


In [ ]:
# ============================================================
# PREREQUISITE CHECK — fail fast if the default lakehouse is not wired up correctly.
# Runs BEFORE any data is touched. Confirms the three things this notebook depends on:
#   1. the accelerator package folder is uploaded to Files/accelerator/fabric_phi_deid/
#   2. the rulebook is at   Files/accelerator/config/deid_rules.yaml
#   3. the silver_* source tables exist in THIS (default) lakehouse
# If any check fails, you get an actionable message instead of a confusing error mid-run.
# ============================================================
import os

_problems = []

# 1 + 2: package folder and config file present in the default lakehouse's Files area.
_pkg_dir  = f"{SRC_PATH}/fabric_phi_deid"
if not os.path.isdir(_pkg_dir):
    _problems.append(f"Missing package folder: {_pkg_dir}  "
                     "-> upload src/fabric_phi_deid/ to Files/accelerator/")
if not os.path.isfile(CONFIG_PATH):
    _problems.append(f"Missing rulebook: {CONFIG_PATH}  "
                     "-> upload config/deid_rules.yaml to Files/accelerator/config/")

# 3: the silver source tables must live in THIS default lakehouse (bronze+silver ran here).
_expected_silver = [
    "silver_dim_patient", "silver_dim_provider", "silver_dim_provider_credential",
    "silver_dim_facility", "silver_fact_claim", "silver_fact_encounter",
    "silver_fact_risk_score", "silver_dim_department", "silver_dim_payer",
    "silver_dim_diagnosis", "silver_dim_procedure", "silver_dim_provider_specialty",
    "silver_bridge_provider_dept",
]
_present = {t.name for t in spark.catalog.listTables()}
_missing_tbls = [t for t in _expected_silver if t not in _present]
if _missing_tbls:
    _problems.append(
        f"{len(_missing_tbls)} silver_* table(s) not found in the default lakehouse: "
        f"{_missing_tbls}. Pin the SAME lakehouse you ran 01/02 in as default, or re-run silver."
    )

if _problems:
    raise RuntimeError(
        "Prerequisite check FAILED — fix before running:\n  - " + "\n  - ".join(_problems)
    )
print("Prerequisite check passed: package, rulebook, and all 13 silver_* tables are present.")


In [ ]:
# --- DEMO SETUP (Option 2, synthetic data only). Remove for production. ---
# Production uses Azure Key Vault over a managed private endpoint (see next cell).
# For the synthetic demo we inject a FIXED high-entropy pepper via an env var so no
# F-SKU capacity / managed private endpoint is required. NEVER print PEPPER.
#
# IMPORTANT: this EXACT value must also be pasted into NB_reidentify in the Vault
# workspace, or break-glass re-identification will not match. It was generated once via:
#     python -c "import secrets; print(secrets.token_urlsafe(48))"
import os
DEMO_PEPPER = "xbSJJefaA60C_s2oNjPKr3t7Z1BQaGv9Go9TH5rse-2kGAZHdVGBwA9mnItp0COf"
os.environ["PHI_DEID_PEPPER"] = DEMO_PEPPER
print("PHI_DEID_PEPPER set:", "PHI_DEID_PEPPER" in os.environ)  # True, never the value

In [ ]:
# --- load the tokenization pepper. NEVER print this value. ---
# Production (Option 1): the pepper lives in Azure Key Vault (public access disabled) and
# is fetched over a Fabric managed private endpoint. Set PHI_DEID_KEYVAULT_URL to the vault.
#   import os; os.environ["PHI_DEID_KEYVAULT_URL"] = "https://<kv-name>.vault.azure.net/"
# Demo/synthetic (Option 2): if no F-SKU capacity / managed private endpoint is available,
# inject the pepper via the PHI_DEID_PEPPER env var instead (synthetic data only, never PHI):
#   import os, secrets; os.environ["PHI_DEID_PEPPER"] = secrets.token_urlsafe(48)
# get_pepper() prefers PHI_DEID_PEPPER when set, otherwise falls back to Key Vault.
PEPPER = get_pepper("phi-deid-pepper")
assert PEPPER, "Pepper must be a non-empty, high-entropy secret."
print("Pepper loaded:", bool(PEPPER))   # prints only True, never the value

In [ ]:
# ============================================================
# Provenance & audit context (metadata only — no data is read here).
# A production de-id run is a compliance event: pin exactly WHICH rulebook version
# is about to be applied, and open a PHI-safe audit log for the run.
# ============================================================
import os, getpass

log = get_audit_logger()

CONFIG_SHA = config_fingerprint(cfg)     # proves which rulebook produced the output
# Bump this whenever the Key Vault pepper is rotated (docs/pepper_rotation_runbook.md).
PEPPER_KEY_VERSION = os.environ.get("PHI_DEID_PEPPER_VERSION", "demo-v1")
try:
    ACTOR = getpass.getuser()
except Exception:
    ACTOR = "fabric-notebook"

log.info(f"run start | engine=v{ENGINE_VERSION} profile={PROFILE} "
         f"config_sha256={CONFIG_SHA[:12]}... pepper_ver={PEPPER_KEY_VERSION} actor={ACTOR}")
print("Config fingerprint (SHA-256):", CONFIG_SHA)
print("Engine version:", ENGINE_VERSION, "| pepper key version:", PEPPER_KEY_VERSION)


In [ ]:
# ============================================================
# PRE-FLIGHT POLICY AUDIT (metadata only — reads SCHEMAS, never data values).
# Before a single PHI value is touched, prove the rulebook still matches the live
# silver schema. This is the "policy preview" that catches schema drift:
#   defaulted -> real column with NO explicit rule -> it will be SUPPRESSED
#                (deny-by-default: safe, but surfaced so a human can confirm intent)
#   missing   -> a rule pointing at a column that no longer exists (dead rule / typo)
# A `missing` rule is a HARD FAIL: the schema moved out from under the rulebook.
# ============================================================
SLV, DEID = "silver_", "silver_deid_"

# Tables whose columns are run through the config-driven engine (deny-by-default).
PHI_TABLES = [
    "dim_patient", "dim_provider", "dim_provider_credential", "dim_facility",
    "fact_claim", "fact_encounter", "fact_risk_score",
]
# Reference tables carrying no PHI -> copied through verbatim.
REFERENCE_TABLES = [
    "dim_department", "dim_payer", "dim_diagnosis", "dim_procedure",
    "dim_provider_specialty", "bridge_provider_dept",
]

_drift = []
print(f"{'table':32s} classified  defaulted   missing")
for tbl in PHI_TABLES:
    cols = spark.table(f"{SLV}{tbl}").columns
    rep = audit_coverage(cfg, PROFILE, tbl, cols)
    flag = "" if rep.is_clean else "   <-- review"
    print(f"{tbl:32s} {len(rep.classified):>10} {len(rep.defaulted):>10} {len(rep.missing):>9}{flag}")
    if rep.defaulted:
        log.info(f"coverage {tbl}: defaulted(->suppress)={rep.defaulted}")
    if rep.missing:
        _drift.append((tbl, rep.missing))
        log.warning(f"coverage {tbl}: MISSING rules (schema drift)={rep.missing}")

assert not _drift, (
    "Rulebook references columns absent from the live silver schema (schema drift). "
    f"Fix config/deid_rules.yaml before de-identifying: {_drift}"
)
print("\nPolicy audit passed: every rule maps to a real column; unclassified "
      "columns will be suppressed (deny-by-default).")


In [ ]:
# ============================================================
# De-identify PHI-bearing tables via the config-driven engine; copy reference
# tables verbatim. Each table is ISOLATED: a failure is recorded and the run
# still reports every table, then re-raises after the summary. Row counts are
# checked input == output (de-id is row-preserving) as an integrity guard.
# ============================================================
def _write(df, name):
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{DEID}{name}"))

manifest_tables = {}   # table -> {columns, input_rows, output_rows} for the audit manifest
failures = {}

for tbl in PHI_TABLES:
    try:
        src = spark.table(f"{SLV}{tbl}")
        in_rows = src.count()
        out = deidentify_table(src, cfg, PROFILE, tbl, PEPPER)
        _write(out, tbl)
        out_rows = spark.table(f"{DEID}{tbl}").count()
        assert in_rows == out_rows, f"row count changed {in_rows}->{out_rows} (expected row-preserving)"
        manifest_tables[tbl] = {"columns": src.columns, "input_rows": in_rows, "output_rows": out_rows}
        log.info(f"deid {DEID}{tbl}: rows={out_rows} profile={PROFILE}")
        print(f"{DEID+tbl:32s} de-identified  {out_rows:>8,} rows")
    except Exception as exc:                       # isolate: record and keep going
        failures[tbl] = repr(exc)
        log.error(f"deid FAILED {tbl}: {exc!r}")
        print(f"{DEID+tbl:32s} FAILED         {exc}")

for tbl in REFERENCE_TABLES:
    try:
        src = spark.table(f"{SLV}{tbl}")
        n = src.count()
        _write(src, tbl)
        manifest_tables[tbl] = {"columns": src.columns, "input_rows": n, "output_rows": n}
        log.info(f"copy {DEID}{tbl}: rows={n} (reference, no PHI)")
        print(f"{DEID+tbl:32s} copied (no PHI) {n:>8,} rows")
    except Exception as exc:
        failures[tbl] = repr(exc)
        log.error(f"copy FAILED {tbl}: {exc!r}")
        print(f"{DEID+tbl:32s} FAILED         {exc}")

assert not failures, f"De-id run had {len(failures)} failed table(s): {list(failures)}"
print("\nSILVER DE-ID complete. Raw PHI never left this notebook.")


In [ ]:
# ============================================================
# Persist a PHI-FREE run manifest FIRST (always recorded, even if the residual gate
# below later blocks publish): a tamper-evident record of WHICH rulebook (config_sha256)
# was applied to WHICH tables by WHOM, when — strategy counts & metadata only, never a
# data value. This is the artifact an auditor asks for and how you prove reproducibility
# / detect drift between runs.
# ============================================================
manifest = build_run_manifest(
    cfg, PROFILE, actor=ACTOR, tables=manifest_tables,
    engine_version=ENGINE_VERSION, pepper_key_version=PEPPER_KEY_VERSION,
)

AUDIT_DIR = "/lakehouse/default/Files/audit"
os.makedirs(AUDIT_DIR, exist_ok=True)
manifest_path = f"{AUDIT_DIR}/deid_run_{manifest.run_id}.json"
write_manifest(manifest, manifest_path)

log.info(f"manifest written run_id={manifest.run_id} tables={len(manifest.tables)} "
         f"config_sha256={manifest.config_sha256[:12]}...")
print("Run id:        ", manifest.run_id)
print("Config sha256: ", manifest.config_sha256)
print("Manifest saved:", manifest_path)
print("\nPer-table strategy mix (metadata only — safe to display and share):")
for t in manifest.tables:
    print(f"  {t.table:30s} rows={t.output_rows:>8,}  {t.strategy_counts}")


In [ ]:
# ============================================================
# POST-RUN RESIDUAL-PHI GATE (defense in depth) — the FINAL publish-blocker before 03b.
# Scan the de-identified OUTPUT for values that STILL look like direct identifiers
# (SSN / phone / email shapes). A clean run must score ZERO. Heuristic backstop, not a
# HIPAA determination — the full assertion suite lives in NB_scorecard.
# ============================================================
residual = {}
for tbl in list(PHI_TABLES) + list(REFERENCE_TABLES):
    hits = scan_spark_dataframe(spark.table(f"{DEID}{tbl}"), sample_limit=10000)
    if hits:
        residual[tbl] = hits

if residual:
    log.error(f"residual PHI patterns detected: {residual}")
    print("RESIDUAL PHI DETECTED (sampled):")
    for tbl, cols in residual.items():
        print(f"  {tbl}: {cols}")
else:
    print("Residual-PHI scan: 0 hits across all de-identified tables (sampled).")

assert not residual, (
    "De-identified output still contains identifier-shaped values. Do NOT publish to gold. "
    f"Offending tables/columns: {residual}"
)


### What just happened
- `MRN` → keyed HMAC token (`PT-…`), identical across every table → **joins still work**.
- `FirstName`/`LastName`/`PatientName` → consistent synthetic values (irreversible).
- `DateOfBirth` / service dates → **year only** (Safe Harbor) or **shifted** (Expert Determination).
- `ZIP` → 3 digits, `000` for low-population prefixes; `Age` capped at 90.
- Any column **not** in the rulebook was **suppressed** (deny-by-default) — nothing leaks by omission.

The crosswalk needed to re-identify (`token → original`) lives ONLY in the restricted
Vault workspace via `NB_reidentify`. Next: build the PHI-free star schema in `03b`.

### How this compares to what others do

This notebook is one *governed crossing point* — a single, auditable place where PHI
becomes de-identified. The design choices below are what separate a production accelerator
from a hand-rolled masking script.

| Capability | Hand-rolled script | MS Presidio | Tonic / Privitar | **This accelerator** |
|---|---|---|---|---|
| Referential integrity across tables (a patient tokenizes identically everywhere → joins survive) | rarely | manual | yes | **yes — keyed HMAC-SHA256 tokens** |
| Deny-by-default (unclassified column → suppressed, never leaks by omission) | no | no | policy-dependent | **yes** |
| Config-driven rulebook, validated & fail-fast before touching data | no | partial | yes | **yes — `validate_config` / `load_rules`** |
| Pre-flight policy audit vs. live schema (catches schema drift) | no | no | some | **yes — `audit_coverage`** |
| PHI-free run manifest (who / what / when / which rulebook `config_sha256`) | no | no | enterprise tier | **yes — `build_run_manifest`** |
| Post-run residual-PHI gate before publish | no | no | yes | **yes — `scan_spark_dataframe`** |
| Secret (pepper) from Key Vault, rotatable, never in code | no | n/a | yes | **yes — `get_pepper`, min-entropy enforced** |
| HIPAA Safe Harbor **and** Expert Determination profiles | no | no | yes | **yes — profile switch** |
| Runs natively on Fabric / Spark at Delta scale | varies | needs glue | separate service | **yes — Spark UDFs over Delta** |

**Where each shines:** Presidio is strongest at *free-text NER* (finding identifiers in
unstructured notes) — complementary, and on the roadmap here as `ner_text.py`. Tonic/Privitar
are mature commercial platforms (and MACC-eligible) when you want a managed service. This
accelerator is the transparent, in-tenant, config-driven reference pattern: no data leaves
Fabric, every run is provable, and the rulebook is reviewable in Git.

> **Not a certified de-id service.** Passing the residual-PHI gate and `NB_scorecard` is
> **necessary but not sufficient** for a HIPAA determination — see
> [`docs/pre_real_phi_checklist.md`](../docs/pre_real_phi_checklist.md).
